In [0]:
df = spark.createDataFrame([("bharath",30,"vizag"), ("arjun",40,"Hyd"), ("sai",50,"vizag")], ["name", "age", "city"])
display(df)


In [0]:
# Write as Delta format
df.write.format("delta").mode("overwrite").save("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta")
    
# Write as Parquet format
df.write.format("parquet").mode("overwrite").save("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_parquet")

###schema changes to four columns

In [0]:
df1 = spark.createDataFrame([("bharath",30,"vizag",9030542539), ("arjun",40,"Hyd",9030542538), ("sai",50,"vizag",9030542537)], ["name", "age", "city","mobile"])
display(df)


###delta supports Schema Enforcement
it will fail if any changes in schema


In [0]:
# Write as Delta format
# spark.conf.set("fs.azure.account.key.bharathdevdatalakep1.dfs.core.windows.net", "<your-storage-account-key>")
df1.write.format("delta").mode("overwrite").save("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta")

##delta supports schema evaolution
it evolve by new schema using merge schem aoption

In [0]:
# Write as Delta format
# spark.conf.set("fs.azure.account.key.bharathdevdatalakep1.dfs.core.windows.net", "<your-storage-account-key>")
df1.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta")

###Parquet will load asusaul if any schema changes also

In [0]:

# Write as Parquet format
df1.write.format("parquet").mode("overwrite").save("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_parquet")
display(df1.limit(10))

###Time travel

In [0]:
%sql
select * from delta.`abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta`

In [0]:
%sql
desc history delta.`abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta`

In [0]:
df = spark.read.format("delta").option("versionAsOf", 0).load("abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta")
display(df)

### CRUD Operations

In [0]:
%sql
select * from delta.`abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta`

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit

dtable = DeltaTable.forPath(spark, "abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta")
dtable.update(condition=col("name") == "bharath", set={"age": lit(32)})

dtable.delete(condition=col("name") == "bharath")

In [0]:
%sql
create schema bharath_dev_adb_p1.employeschema

In [0]:
%sql
create table if not exists bharath_dev_adb_p1.employeschema.employee
as
select * from delta.`abfss://test@bharathdevdatalakep1.dfs.core.windows.net/employes_delta`

In [0]:
%sql
select * from  bharath_dev_adb_p1.employeschema.employee

In [0]:
%sql
select * from bharath_dev_adb_p1.employeschema.employee@V0